In [23]:
with open("../../data/tiny_shakespeare.txt", "r", encoding="utf-8") as f:
    text =  f.read()

In [24]:
print(f"length of text: {len(text)}")

length of text: 1115393


In [25]:
print(text[:400])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it 


In [26]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(f"chars: {"".join(chars)}")
print(f"size: {vocab_size}")

chars: 
 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
size: 65


In [27]:
## will try to change using the tokenization code that I made
stoi = {ch:i for i,ch in enumerate(chars)}
itos = {i:ch for i,ch in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: "".join([itos[i] for i in l])

print(encode("hi there"))
print(decode(encode("hi there")))

[46, 47, 1, 58, 46, 43, 56, 43]
hi there


In [28]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:300])

torch.Size([1115393]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [29]:
n = int(len(data) * 0.9)

train = data[:n]
test = data[n:]

In [30]:
block_size = 8
train[:block_size + 1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [31]:
x = train[:block_size]
y = train[1:block_size + 1]

for t in range(block_size):
    context = x[:t+1]
    target = y[t]

    print(f"context: {context}, target: {target}")

context: tensor([18]), target: 47
context: tensor([18, 47]), target: 56
context: tensor([18, 47, 56]), target: 57
context: tensor([18, 47, 56, 57]), target: 58
context: tensor([18, 47, 56, 57, 58]), target: 1
context: tensor([18, 47, 56, 57, 58,  1]), target: 15
context: tensor([18, 47, 56, 57, 58,  1, 15]), target: 47
context: tensor([18, 47, 56, 57, 58,  1, 15, 47]), target: 58


In [32]:
torch.manual_seed(1337)

batch_size = 4
block_size = 8

def get_batch(splits):

    data = train if splits == "train" else test
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix]) # size is batch_size x block_size
    y = torch.stack([data[i+1:i+1+block_size] for i in ix])

    return x, y

xb, yb = get_batch("train")
print("inputs")
print(xb.shape)
print(xb)

print("targets")
print(yb.shape)
print(yb)

for b in range(batch_size):
    for t in range(block_size):
        context = xb[b, :t+1]
        target = yb[b, t]
        print(f"context: {context}, target: {target}")        

inputs
torch.Size([4, 8])
tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])
targets
torch.Size([4, 8])
tensor([[59,  6,  1, 58, 56, 47, 40, 59],
        [43, 43, 54,  1, 47, 58,  1, 58],
        [52, 45, 43, 50, 53,  8,  0, 26],
        [39,  1, 46, 53, 59, 57, 43,  0]])
context: tensor([53]), target: 59
context: tensor([53, 59]), target: 6
context: tensor([53, 59,  6]), target: 1
context: tensor([53, 59,  6,  1]), target: 58
context: tensor([53, 59,  6,  1, 58]), target: 56
context: tensor([53, 59,  6,  1, 58, 56]), target: 47
context: tensor([53, 59,  6,  1, 58, 56, 47]), target: 40
context: tensor([53, 59,  6,  1, 58, 56, 47, 40]), target: 59
context: tensor([49]), target: 43
context: tensor([49, 43]), target: 43
context: tensor([49, 43, 43]), target: 54
context: tensor([49, 43, 43, 54]), target: 1
context: tensor([49, 43, 43, 54,  1]), target: 47
context: tensor([4

In [33]:
print(xb)

tensor([[53, 59,  6,  1, 58, 56, 47, 40],
        [49, 43, 43, 54,  1, 47, 58,  1],
        [13, 52, 45, 43, 50, 53,  8,  0],
        [ 1, 39,  1, 46, 53, 59, 57, 43]])


In [40]:
import torch 
import torch.nn as nn 
from torch.nn import functional as F 

torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets):

        # idx and target are both (B, T) tensor of integers
        logits = self.token_embedding_table(idx) # (B, T, C), batch, time, channel : vocab size

        B, T, C = logits.shape 
        logits = logits.view(B*T, C)
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)

        return logits, loss
    
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss)

torch.Size([32, 65])
tensor(4.8948, grad_fn=<NllLossBackward0>)
